In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
hour_data = pd.read_csv("../data/processed/hour_features.csv")

In [3]:
hour_data.dtypes

instant          int64
dteday             str
season           int64
yr               int64
mnth             int64
hr               int64
holiday          int64
weekday          int64
workingday       int64
weathersit       int64
temp           float64
atemp          float64
hum            float64
windspeed      float64
casual           int64
registered       int64
cnt              int64
time_of_day        str
high_demand      int64
rush_hour        int64
dtype: object

In [4]:
def missing_values(data):
    count=0
    for row in data:
        for value in row:
            if value=="":
                count+=1
    return count
print(missing_values(hour_data))

0


We will study for KNN algorithm. For project target is 'cnt'. Also we have X(info that model gets), y-what model should predict. But 'registered' and 'casual' are straight linked with 'cnt' so if we gonna predict 'cnt' these 2 variable will give answer to us. It is called 'data leakage'. High_demand is also used in 'cnt' so we can't use it in features.

So we need to remove 'registered', 'casual', 'high_demand'
We have categorical features such as - "season", "yr", "mnth", "hr", "holiday", "weekday ""workingday", "weathersit", "time_of_day"

Numerical - "temp","atemp","hum","windspeed", "rush_hour"
        

In [5]:
categorical = [
        "season",
        "yr",
        "mnth",
        "hr",
        "holiday",
        "weekday",
        "workingday",
        "weathersit",
        "time_of_day"
    ]
numerical = [
        "temp",
        "atemp",
        "hum",
        "windspeed",
        "rush_hour"
    ]

In [6]:
y=hour_data["cnt"]

drop_features=[
    "cnt",
    "casual",
    "registered",
    "high_demand",
    "dteday", 
    "instant"
]

X=hour_data.drop(columns=drop_features)
print("X columns: ")
print(X.columns)
print("\ny:", y.name)

X columns: 
Index(['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
       'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'time_of_day',
       'rush_hour'],
      dtype='str')

y: cnt


In [7]:
print("X shape:", X.shape)
print("y shape", y.shape)
print(len(X) == len(y))

X shape: (17379, 14)
y shape (17379,)
True


Encoding we use to transform categorical data to numerical representation.
Scaling we use to do all in one scale, to calculate distance well. 

In [8]:
def one_hot_encode(data, column):
    categories=[]
    for value in data[column]:
        if value not in categories:
            categories.append(value)
    result=data.copy()

    for category in categories:
        result[f"{column}_{category}"] = [1 if value == category else 0 for value in data[column]]    
    result=result.drop(columns=[column])
    return result

def encode_cat(data, categorical_columns):
    result=data.copy()
    for column in categorical_columns:
        result=one_hot_encode(result, column)
    return result

In [9]:
def means(values):
    return sum(values)/len(values)

In [10]:
def standart_dev(values):
    avg=means(values)

    squared_diff=[]
    for value in values:
        diff=value-avg
        squared_diff.append(diff**2)
    variance=sum(squared_diff)/len(values)
    return variance**0.5

In [11]:
def standart_scale(values):
    avg=means(values)
    std=standart_dev(values)

    if std==0:
        return [0 for value in values]
    return [(value-avg)/std for value in values]

In [12]:
def scaling(data, numerical_columns):
    result=data.copy()
    for column in numerical_columns:
        result[column]=standart_scale(data[column].tolist())
    return result

In [13]:
X_encoded=encode_cat(X, categorical)
X_final=scaling(X_encoded, numerical)

In [14]:
def train_test_set(X, y, test_ratio=0.2):
    split_index=int(len(X)*(1-test_ratio))

    X_train=X.iloc[:split_index]
    X_test=X.iloc[split_index:]

    y_train=y.iloc[:split_index]
    y_test=y.iloc[split_index:]
    return X_train, X_test, y_train, y_test

In [15]:
X_train, X_test, y_train, y_test=train_test_set(X_final, y, test_ratio=0.2)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)